# Multimodal CVD Early Risk Prediction — Leakage-Free Preprocessing Pipeline (Kaggle Ready)

**Authors:** Mohammad Nafis Sadiq (`232-15-895`) & Rayan Ahmed (`232-15-847`) | **Supervisor:** Md. Mizanur Rahman (DIU CSE499)  
**Governed By:** `STEP-019`–`STEP-022` (`OS §4.4–§4.5`, `DEC0004`, `DEC0005`, `DEC0006`, `DEC0007`)

### Strict Scientific Rules Enforced Inside This Notebook:
1. **Zero Cross-Cohort Stitching (`DEC0004`):** `PTB-XL`, `CARDIO-ML-ED`, `CAIR-CVD-2025`, and `Kaggle sulianova` represent distinct individuals; every patient stays strictly in their cohort with a **4D Modality Presence Mask (`[has_C, has_E, has_L, has_S]`)**.
2. **Stage A vs Stage B Wall (`OS §4.4`):**
   - **Stage A (Record-Local Pre-Split):** Unit conversions (`age_days / 365.25`), pediatric `<18y` & outlier quarantine (`ecg_id=12722`, `8` JPG strips, `SBP<=DBP`, `LDL<0`), digital ECG filtering (`0.5 Hz` high-pass + `50 Hz` notch), and scanned PNG `90°` CCW rotation + red-grid suppression.
   - **Patient-Grouped Split (`STEP-020`):** Official `PTB-XL strat_fold 1-8 / 9 / 10` + `StratifiedGroupKFold` / grouped split (`Col1` in `CARDIO-ML-ED`). **Final Test set sealed before fitting anything!**
   - **Stage B (Train-Only Fitted):** `_isnan` binary MNAR missingness masks + `IterativeImputer` / Median + `RobustScaler` + `OneHotEncoder(handle_unknown='ignore')` fitted **exclusively on Train**, serialized with `FitStateHash`. **Zero pre-split SMOTE!**
3. **PROBAST Domain 3 (`Tier A` Primary):** Excludes `Troponine - I (Q)`, `CVD Risk Score`, and `Blood Pressure Category` from primary predictor tensors.


## Cell 1 — Environment Setup & Automatic Kaggle / Local Path Discovery
Works identically whether executed on **Kaggle Cloud (`/kaggle/input/...`)** or **locally (`e:/FYDP/Dataset/...`)**.


In [ ]:
import ast, collections, hashlib, json, math, os, random, re, struct, warnings, zipfile
import xml.etree.ElementTree as ET
from pathlib import Path

import cv2
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import signal
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.model_selection import StratifiedGroupKFold, StratifiedShuffleSplit
from sklearn.preprocessing import OneHotEncoder, RobustScaler

warnings.filterwarnings("ignore", category=UserWarning)
SEED = 20261127
np.random.seed(SEED)
random.seed(SEED)

IS_KAGGLE = os.path.isdir("/kaggle/input")
WORK_DIR = Path("/kaggle/working/preprocessed_outputs") if IS_KAGGLE else Path("../processed_kaggle_preview")
WORK_DIR.mkdir(parents=True, exist_ok=True)

def discover_dataset_paths():
    roots = [Path("/kaggle/input")] if IS_KAGGLE else [Path("../Dataset"), Path("Dataset"), Path("e:/FYDP/Dataset")]
    found = {"ptbxl": None, "cardiomled": None, "cair": None, "sulianova": None}
    for root in roots:
        if not root.exists():
            continue
        for p in root.rglob("ptbxl_database.csv"):
            found["ptbxl"] = p.parent
        for p in root.rglob("Datasheet1 (Edited).csv"):
            found["cardiomled"] = p.parent
        for p in root.rglob("CVD Dataset.csv"):
            found["cair"] = p.parent
        for p in root.rglob("cardio_train.csv"):
            found["sulianova"] = p.parent
    return found

DATA_PATHS = discover_dataset_paths()
print(f"Environment: {'KAGGLE' if IS_KAGGLE else 'LOCAL'} | Output Dir: {WORK_DIR}")
for k, v in DATA_PATHS.items():
    print(f"  {k:12s} -> {v}")


## Cell 2 — Stage A Helpers (XLSX OpenXML Parser, Schema Leakage Guard & ECG DSP)
- Note: `CARDIO-ML-ED/Datasheet1 (Edited).csv` is actually an **Excel `.xlsx` workbook** disguised with a `.csv` extension! We parse `Sheet 1` safely cleanly here.
- Also defines **50 Hz biquad notch** ($Q=30$) + **0.5 Hz zero-phase Butterworth high-pass** + **Red Millimeter Grid Suppression** for `.png` ECG scans.


In [ ]:
PROHIBITED_LEAK_COLS = {
    "cvd risk score", "blood pressure category", "initial diagnosis", "final diagnosis",
    "death", "cause of death", "patient status on discharge", "method of referral",
    "scp_codes", "ecg labels", "cardio", "cvd risk level", "troponine - i (q)"
}

def assert_no_leak_features(cols, allow_troponin_tier_b=False):
    for c in cols:
        low = c.strip().lower()
        if low == "troponine - i (q)" and allow_troponin_tier_b:
            continue
        if low in PROHIBITED_LEAK_COLS:
            raise ValueError(f"PROBAST / LEAKAGE VIOLATION: Prohibited predictor '{c}' detected!")

def read_cardiomled_xlsx_sheet1(xlsx_path: Path) -> pd.DataFrame:
    with zipfile.ZipFile(xlsx_path) as z:
        sst = []
        if "xl/sharedStrings.xml" in z.namelist():
            root = ET.fromstring(z.read("xl/sharedStrings.xml"))
            for si in root.findall("{http://schemas.openxmlformats.org/spreadsheetml/2006/main}si"):
                sst.append("".join(t.text or "" for t in si.iter("{http://schemas.openxmlformats.org/spreadsheetml/2006/main}t")))
        root = ET.fromstring(z.read("xl/worksheets/sheet1.xml"))
        rows = []
        for row in root.findall(".//{http://schemas.openxmlformats.org/spreadsheetml/2006/main}row"):
            r_dict = {}
            for c in row.findall("{http://schemas.openxmlformats.org/spreadsheetml/2006/main}c"):
                ref = c.attrib.get("r", "")
                idx = 0
                for ch in (ch for ch in ref if ch.isalpha()):
                    idx = idx * 26 + (ord(ch.upper()) - ord("A") + 1)
                t = c.attrib.get("t", "")
                v_el = c.find("{http://schemas.openxmlformats.org/spreadsheetml/2006/main}v")
                is_el = c.find("{http://schemas.openxmlformats.org/spreadsheetml/2006/main}is")
                val = ""
                if v_el is not None and v_el.text is not None:
                    val = sst[int(v_el.text)] if t == "s" else v_el.text
                elif is_el is not None:
                    val = "".join(x.text or "" for x in is_el.iter("{http://schemas.openxmlformats.org/spreadsheetml/2006/main}t"))
                if idx > 0:
                    r_dict[idx] = val
            if r_dict:
                rows.append([r_dict.get(i, "") for i in range(1, max(r_dict) + 1)])
    hdr = [h.strip() for h in rows[0]]
    unique_hdr = ["SerialNo", "PatientFileNo"] + hdr[2:]
    body = [[str(x).strip() for x in (r + [""] * (len(unique_hdr) - len(r)))] for r in rows[1:] if any(str(x).strip() for x in r)]
    return pd.DataFrame(body, columns=unique_hdr)

def clean_wfdb_12lead_signal(sig_12xT: np.ndarray, fs_hz: float = 500.0) -> np.ndarray:
    """Record-local ST-T morphology-safe filter: 0.5 Hz zero-phase Butterworth HP + 50 Hz notch."""
    sos_hp = signal.butter(2, 0.5, btype="highpass", fs=fs_hz, output="sos")
    out = signal.sosfiltfilt(sos_hp, sig_12xT.astype(np.float64), axis=-1)
    if fs_hz > 120.0:  # Safely bypass 50 Hz notch when fs=100 Hz (Nyquist edge = 50 Hz!)
        b_n, a_n = signal.iirnotch(w0=50.0, Q=30.0, fs=fs_hz)
        out = signal.filtfilt(b_n, a_n, out, axis=-1)
    # Record-local standardized mV clip preserving inter-lead relative amplitude
    std = np.std(out, axis=-1, keepdims=True)
    mean = np.mean(out, axis=-1, keepdims=True)
    out = (out - mean) / np.where(std > 1e-6, std, 1.0)
    return np.clip(out, -10.0, 10.0).astype(np.float32)

def clean_cardiomled_png_scan(img_path: Path, target_hw=(256, 768)) -> np.ndarray:
    """Rotate portrait 90 deg CCW, remove red millimeter paper grid, and letterbox resize cleanly."""
    bgr = cv2.imread(str(img_path))
    if bgr is None:
        raise ValueError(f"Unreadable ECG PNG: {img_path}")
    h, w = bgr.shape[:2]
    if h > w:
        bgr = cv2.rotate(bgr, cv2.ROTATE_90_COUNTERCLOCKWISE)
    # Red-grid suppression in float RGB space
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB).astype(np.float32)
    r, g, b = rgb[:, :, 0], rgb[:, :, 1], rgb[:, :, 2]
    red_excess = np.maximum(0.0, r - np.maximum(g, b))
    gray_clean = np.clip(0.299 * r + 0.587 * g + 0.114 * b + 0.75 * red_excess, 0.0, 255.0)
    chroma_max = np.maximum(r, np.maximum(g, b))
    gray_clean[(gray_clean >= 155.0) & (chroma_max >= 165.0)] = 255.0
    # Aspect-preserving letterbox padding to white background (255)
    th, tw = target_hw
    ch, cw = gray_clean.shape[:2]
    scale = min(tw / cw, th / ch)
    nw, nh = max(1, int(cw * scale)), max(1, int(ch * scale))
    resized = cv2.resize(gray_clean.astype(np.uint8), (nw, nh), interpolation=cv2.INTER_AREA)
    canvas = np.full((th, tw), 255, dtype=np.uint8)
    y0, x0 = (th - nh) // 2, (tw - nw) // 2
    canvas[y0 : y0 + nh, x0 : x0 + nw] = resized
    # Return normalized inverted ink intensity [0.0=paper background, 1.0=dark ECG ink]
    return ((255.0 - canvas.astype(np.float32)) / 255.0)


## Cell 3 — Stage A: Deterministic Cleaning & Quarantine Per Cohort (`No Dataset Statistics Fitted!`)
Runs exact record-local cleaning across whichever cohorts are mounted cleanly (`DS0001–DS0004`).


In [ ]:
def prepare_ptbxl_stage_a(ptb_dir: Path) -> pd.DataFrame:
    scp_df = pd.read_csv(ptb_dir / "scp_statements.csv", index_col=0)
    diag_map = scp_df[scp_df["diagnostic"] == 1.0]["diagnostic_class"].to_dict()
    df = pd.read_csv(ptb_dir / "ptbxl_database.csv")
    # Quarantine STEP-013 defect + pediatric/missing-age
    df = df[df["ecg_id"] != 12722].copy()
    df["age_years"] = pd.to_numeric(df["age"], errors="coerce")
    df = df[df["age_years"] >= 18.0].copy()
    def _extract_target(s):
        codes = ast.literal_eval(s)
        cls = {diag_map[k] for k, v in codes.items() if k in diag_map and float(v) > 0}
        if not cls:
            return np.nan
        return 0.0 if cls == {"NORM"} else 1.0
    df["target"] = df["scp_codes"].apply(_extract_target)
    df = df.dropna(subset=["target"]).copy()
    df["target"] = df["target"].astype(int)
    df["PatientGroupID"] = df["patient_id"].astype(int).astype(str)
    df["sex_bin"] = df["sex"].map({1: "F", 0: "M"})
    df["height_cm"] = pd.to_numeric(df["height"], errors="coerce").where(lambda x: x.between(100, 220))
    df["weight_kg"] = pd.to_numeric(df["weight"], errors="coerce").where(lambda x: x.between(30, 200))
    df["bmi_kgm2"] = (df["weight_kg"] / ((df["height_cm"] / 100.0) ** 2)).where(lambda x: x.between(10, 70))
    return df

def prepare_cardiomled_stage_a(cm_dir: Path) -> pd.DataFrame:
    import csv
    df = read_cardiomled_xlsx_sheet1(cm_dir / "Datasheet1 (Edited).csv")
    with open(cm_dir / "ECG Labels.csv", encoding="utf-8") as f:
        lbl_set = {r[0].strip() for r in csv.reader(f) if len(r) >= 2 and r[0].strip()}
    ecg_dir = cm_dir / "ECGS"
    img_by_base = collections.defaultdict(list)
    if ecg_dir.exists():
        for p in sorted(ecg_dir.iterdir()):
            if p.is_file():
                st = re.sub(r"\((\d+)\)$", r"-\1", p.stem.strip())
                base = st.split("-")[0]
                img_by_base[base].append((p, st))

    typo_stems = {"140354", "148679"}
    keep_rows = []
    for _, r in df.iterrows():
        age = pd.to_numeric(r["Patient Age"], errors="coerce")
        init_d = str(r["Initial diagnosis"]).strip()
        if pd.isna(age) or age < 18.0 or init_d not in ("Cardiac", "Noncardiac"):
            continue
        pid = str(r["PatientFileNo"]).strip()
        cands = img_by_base.get(pid, [])
        # Quarantine JPG 6-lead strips & typo discrepancies per RISK0012
        if any(p.suffix.lower() != ".png" or st in typo_stems for p, st in cands):
            continue
        pngs = [str(p) for p, st in cands if st in lbl_set]
        keep_rows.append({
            "PatientGroupID": pid,
            "age_years": float(age),
            "sex_bin": str(r["Patient Gender"]).strip() if r["Patient Gender"] in ("F", "M") else np.nan,
            "sbp_mmhg": pd.to_numeric(r["Systolic blood pressure"], errors="coerce"),
            "dbp_mmhg": pd.to_numeric(r["Diastolic blood pressure"], errors="coerce"),
            "pr_bpm": pd.to_numeric(r["Pulse rate (PR)"], errors="coerce"),
            "creatinine_mgdl": pd.to_numeric(r["Creatinine"], errors="coerce"),
            "wbc_ul": pd.to_numeric(r["WBC"], errors="coerce"),
            "png_path": pngs[0] if pngs else None,
            "has_ecg": int(len(pngs) > 0),
            "target": 1 if init_d == "Cardiac" else 0,
        })
    return pd.DataFrame(keep_rows)

def prepare_cair_stage_a(cair_dir: Path) -> pd.DataFrame:
    df = pd.read_csv(cair_dir / "CVD Dataset.csv")
    # Quarantine inverted BP (SBP <= DBP) and negative Friedewald LDL (< 0)
    bad_bp = (df["Systolic BP"].notna()) & (df["Diastolic BP"].notna()) & (df["Systolic BP"] <= df["Diastolic BP"])
    bad_ldl = (df["Estimated LDL (mg/dL)"].notna()) & (df["Estimated LDL (mg/dL)"] < 0.0)
    df = df[~(bad_bp | bad_ldl)].copy()
    # Hard drop target leakage columns CVD Risk Score & Blood Pressure Category!
    df["PatientGroupID"] = [f"CAIR_{i}" for i in range(len(df))]
    df["age_years"] = pd.to_numeric(df["Age"], errors="coerce")
    df = df[df["age_years"].isna() | (df["age_years"] >= 18.0)].copy()
    df["sex_bin"] = df["Sex"].where(df["Sex"].isin(["F", "M"]))
    df["bmi_kgm2"] = pd.to_numeric(df["BMI"], errors="coerce")
    df["sbp_mmhg"] = pd.to_numeric(df["Systolic BP"], errors="coerce")
    df["dbp_mmhg"] = pd.to_numeric(df["Diastolic BP"], errors="coerce")
    df["tc_mgdl"] = pd.to_numeric(df["Total Cholesterol (mg/dL)"], errors="coerce")
    df["hdl_mgdl"] = pd.to_numeric(df["HDL (mg/dL)"], errors="coerce")
    df["ldl_mgdl"] = pd.to_numeric(df["Estimated LDL (mg/dL)"], errors="coerce")
    df["fbs_mgdl"] = pd.to_numeric(df["Fasting Blood Sugar (mg/dL)"], errors="coerce")
    df["smoke_bin"] = df["Smoking Status"].map({"Y": 1.0, "N": 0.0})
    df["fam_hx_bin"] = df["Family History of CVD"].map({"Y": 1.0, "N": 0.0})
    df["activity_lvl"] = df["Physical Activity Level"]
    df["target"] = (df["CVD Risk Level"] == "HIGH").astype(int)
    return df

def prepare_sulianova_stage_a(sul_dir: Path) -> pd.DataFrame:
    df = pd.read_csv(sul_dir / "cardio_train.csv", sep=";")
    feat_cols = ["age", "gender", "height", "weight", "ap_hi", "ap_lo", "cholesterol", "gluc", "smoke", "alco", "active"]
    # Remove discordant duplicates (conflicting cardio outcome for identical features) & exact dups
    nunique_y = df.groupby(feat_cols)["cardio"].transform("nunique")
    df = df[nunique_y == 1].drop_duplicates(subset=feat_cols + ["cardio"]).copy()
    df["age_years"] = df["age"] / 365.25
    df["bmi_kgm2"] = df["weight"] / ((df["height"] / 100.0) ** 2)
    valid = (
        df["ap_hi"].between(50, 250) & df["ap_lo"].between(30, 180) & (df["ap_hi"] > df["ap_lo"])
        & df["height"].between(100, 220) & df["weight"].between(30, 200) & df["bmi_kgm2"].between(10, 70)
    )
    df = df[valid].copy()
    df["PatientGroupID"] = df["id"].astype(str)
    df["sex_bin"] = df["gender"].map({1: "F", 2: "M"})
    df["sbp_mmhg"] = df["ap_hi"].astype(float)
    df["dbp_mmhg"] = df["ap_lo"].astype(float)
    df["target"] = df["cardio"].astype(int)
    return df

COHORTS_A = {}
if DATA_PATHS["ptbxl"]: COHORTS_A["DS0001_ptbxl"] = prepare_ptbxl_stage_a(DATA_PATHS["ptbxl"])
if DATA_PATHS["cardiomled"]: COHORTS_A["DS0002_cardiomled"] = prepare_cardiomled_stage_a(DATA_PATHS["cardiomled"])
if DATA_PATHS["cair"]: COHORTS_A["DS0003_cair"] = prepare_cair_stage_a(DATA_PATHS["cair"])
if DATA_PATHS["sulianova"]: COHORTS_A["DS0004_sulianova"] = prepare_sulianova_stage_a(DATA_PATHS["sulianova"])

for name, d in COHORTS_A.items():
    print(f"{name:20s}: eligible_rows={len(d):6d} | unique_patients={d['PatientGroupID'].nunique():6d} | CVD+={d['target'].mean()*100:.2f}%")


## Cell 4 — STEP-020 Preview: Patient-Grouped Stratified Splitting (`Zero Patient Overlap Proof`)
Before **ANY** imputer or scaler touches a single number, split cleanly by `PatientGroupID`:
- `PTB-XL`: Preserves official `strat_fold` (`1–8` Train, `9` Val, `10` Test).
- Other cohorts: Grouped stratified split (`70%` Train / `15%` Val / `15%` Test) keeping repeated patient visits (`Col1`) intact in one partition.


In [ ]:
def patient_grouped_split(df: pd.DataFrame, group_col="PatientGroupID", y_col="target", seed=SEED):
    if "strat_fold" in df.columns:
        tr = df[df["strat_fold"].isin(range(1, 9))].copy()
        va = df[df["strat_fold"] == 9].copy()
        te = df[df["strat_fold"] == 10].copy()
    else:
        sgkf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=seed)
        folds = np.zeros(len(df), dtype=int)
        for f_idx, (_, val_idx) in enumerate(sgkf.split(df, df[y_col], df[group_col])):
            folds[val_idx] = f_idx
        tr = df[folds < 7].copy()
        va = df[folds == 7].copy()
        te = df[folds >= 8].copy()

    # Hard Zero-Overlap Assertion (OS §4.4 / G07 invariant)
    p_tr, p_va, p_te = set(tr[group_col]), set(va[group_col]), set(te[group_col])
    assert len(p_tr & p_va) == 0 and len(p_tr & p_te) == 0 and len(p_va & p_te) == 0, "PATIENT LEAK!"
    return tr, va, te

SPLITS = {}
for name, d in COHORTS_A.items():
    tr, va, te = patient_grouped_split(d)
    SPLITS[name] = (tr, va, te)
    print(f"{name:20s}: Train={len(tr):6d} | Val={len(va):5d} | Test={len(te):5d} | zero_overlap=PASS")


## Cell 5 — Stage B: Train-Only Imputation (`IterativeImputer` + MNAR `_isnan`), `RobustScaler` & `OneHotEncoder`
Fits preprocessing **strictly on `df_train`**, freezes `FitStateHash` (SHA-256), and transforms `Val` & `Test` cleanly (`0` test statistics touched!). Also computes `pos_weight` on `y_train` (`no pre-split SMOTE`).


In [ ]:
class TrainOnlyPipelineWrapper:
    def __init__(self, num_cols, cat_cols=()):
        assert_no_leak_features(list(num_cols) + list(cat_cols), allow_troponin_tier_b=False)
        self.num_cols = list(num_cols)
        self.cat_cols = list(cat_cols)
        self.imputer = IterativeImputer(max_iter=15, random_state=SEED, initial_strategy="median") if self.num_cols else None
        self.scaler = RobustScaler() if self.num_cols else None
        self.encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False) if self.cat_cols else None
        self.fit_state_hash_ = None

    def fit(self, df_train: pd.DataFrame):
        if self.num_cols:
            X_num = df_train[self.num_cols].astype(np.float64).values
            X_imp = self.imputer.fit_transform(X_num)
            self.scaler.fit(X_imp)
        if self.cat_cols:
            X_cat = df_train[self.cat_cols].fillna("MISSING").astype(str).values
            self.encoder.fit(X_cat)
        payload = {"center": self.scaler.center_.tolist() if self.num_cols else [], "scale": self.scaler.scale_.tolist() if self.num_cols else []}
        self.fit_state_hash_ = hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()
        return self

    def transform(self, df_part: pd.DataFrame) -> np.ndarray:
        blocks = []
        if self.num_cols:
            raw_num = df_part[self.num_cols].astype(np.float64).values
            isnan_mask = np.isnan(raw_num).astype(np.float32)  # Preserve MNAR ordering signal!
            scaled = self.scaler.transform(self.imputer.transform(raw_num)).astype(np.float32)
            blocks.extend([scaled, isnan_mask])
        if self.cat_cols:
            raw_cat = df_part[self.cat_cols].fillna("MISSING").astype(str).values
            blocks.append(self.encoder.transform(raw_cat).astype(np.float32))
        return np.hstack(blocks)

COHORT_SCHEMAS = {
    "DS0001_ptbxl": (["age_years", "height_cm", "weight_kg", "bmi_kgm2"], ["sex_bin"]),
    "DS0002_cardiomled": (["age_years", "sbp_mmhg", "dbp_mmhg", "pr_bpm", "creatinine_mgdl", "wbc_ul"], ["sex_bin"]),
    "DS0003_cair": (["age_years", "bmi_kgm2", "sbp_mmhg", "dbp_mmhg", "tc_mgdl", "hdl_mgdl", "ldl_mgdl", "fbs_mgdl", "smoke_bin", "fam_hx_bin"], ["sex_bin", "activity_lvl"]),
    "DS0004_sulianova": (["age_years", "height", "weight", "bmi_kgm2", "sbp_mmhg", "dbp_mmhg", "cholesterol", "gluc", "smoke", "alco", "active"], ["sex_bin"]),
}

PROCESSED_ARRAYS = {}
for name, (tr, va, te) in SPLITS.items():
    num_c, cat_c = COHORT_SCHEMAS[name]
    pipe = TrainOnlyPipelineWrapper(num_c, cat_c).fit(tr)
    X_tr, X_va, X_te = pipe.transform(tr), pipe.transform(va), pipe.transform(te)
    assert not np.isnan(X_tr).any() and not np.isnan(X_va).any() and not np.isnan(X_te).any()
    pos_w = float((tr["target"] == 0).sum() / max(1, (tr["target"] == 1).sum()))
    joblib.dump(pipe, WORK_DIR / f"{name}_train_fitted_pipe.joblib")
    np.savez_compressed(WORK_DIR / f"{name}_tabular_splits.npz", X_train=X_tr, y_train=tr["target"].values, X_val=X_va, y_val=va["target"].values, X_test=X_te, y_test=te["target"].values)
    PROCESSED_ARRAYS[name] = (X_tr.shape, pos_w, pipe.fit_state_hash_)
    print(f"{name:20s}: X_train={X_tr.shape}, train_pos_weight={pos_w:.3f}, FitStateHash={pipe.fit_state_hash_[:12]}...")


## Cell 6 — ECG Visual Verification: Raw vs Filtered Waveforms (`PTB-XL`) & Paper Red-Grid Removal (`CARDIO-ML-ED`)
Inspects **ST-T morphology fidelity** (`r >= 0.96`) on sample digital WFDB waveforms (`500 Hz`) and **cropped PNG grid suppression** (`90°` CCW + ink isolation).


In [ ]:
if DATA_PATHS["ptbxl"]:
    # Read one sample 500 Hz WFDB record directly in pure Python + NumPy (works even without wfdb package!)
    ptb_dir = DATA_PATHS["ptbxl"]
    row0 = COHORTS_A["DS0001_ptbxl"].iloc[0]
    dat_bytes = (ptb_dir / (row0["filename_hr"] + ".dat")).read_bytes()
    raw_hr = np.frombuffer(dat_bytes, dtype="<i2").reshape(-1, 12).T / 1000.0  # shape (12, 5000) in mV
    filt_hr = clean_wfdb_12lead_signal(raw_hr, fs_hz=500.0)
    corr_lead_ii = np.corrcoef(raw_hr[1, 250:4750], filt_hr[1, 250:4750])[0, 1]
    print(f"Sample PTB-XL HR Lead II correlation (Raw vs Filtered 0.5Hz HP + 50Hz Notch): r = {corr_lead_ii:.5f}")

    fig, axes = plt.subplots(2, 1, figsize=(12, 4), sharex=True)
    t_sec = np.arange(2000) / 500.0
    axes[0].plot(t_sec, raw_hr[1, :2000], lw=0.9)
    axes[0].set_title("Raw PTB-XL 500 Hz Lead II (mV)")
    axes[1].plot(t_sec, filt_hr[1, :2000], lw=0.9, color="tab:green")
    axes[1].set_title("Cleaned Lead II (0.5 Hz Butterworth HP + 50 Hz Notch — ST-T Preserved)")
    axes[1].set_xlabel("Time (s)")
    plt.tight_layout()
    plt.show()

if DATA_PATHS["cardiomled"] and "DS0002_cardiomled" in COHORTS_A:
    cm_imgs = COHORTS_A["DS0002_cardiomled"].dropna(subset=["png_path"])
    if len(cm_imgs) > 0:
        sample_png = Path(cm_imgs.iloc[0]["png_path"])
        raw_bgr = cv2.imread(str(sample_png))
        if raw_bgr.shape[0] > raw_bgr.shape[1]:
            raw_bgr = cv2.rotate(raw_bgr, cv2.ROTATE_90_COUNTERCLOCKWISE)
        cleaned_ink = clean_cardiomled_png_scan(sample_png, target_hw=(256, 768))

        fig, axes = plt.subplots(2, 1, figsize=(12, 5))
        axes[0].imshow(cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2RGB))
        axes[0].set_title(f"Rotated Raw Paper Scan ({sample_png.name}) — Red Grid Visible")
        axes[1].imshow(cleaned_ink, cmap="gray")
        axes[1].set_title("Cleaned & Letterboxed (256x768) — Red Grid Suppressed, Beats Normalized")
        plt.tight_layout()
        plt.show()


## Cell 7 — Multimodal 4D Modality Presence Mask (`[has_C, has_E, has_L, has_S]`) Readiness Summary
Verifies exact cohort modality masks ready for **Missing-Modality Transformer Cross-Attention** on Kaggle GPU (`Stage S10–S11`).


In [ ]:
MODALITY_MASK_BY_COHORT = {
    "DS0001_ptbxl": [1, 1, 0, 0],       # Clinical partial + Digital ECG (0 Labs, 0 Lifestyle)
    "DS0002_cardiomled": [1, 1, 1, 0],  # 329 same-patient C+E_png+L ([1,0,1,0] when has_ecg==0)
    "DS0003_cair": [1, 0, 1, 1],        # BD Clinical + Lab + Lifestyle (0 ECG)
    "DS0004_sulianova": [1, 0, 1, 1],   # 68.5K Clinical + Ordinal Lab + Lifestyle (0 ECG)
}
for k, mask in MODALITY_MASK_BY_COHORT.items():
    if k in COHORTS_A:
        print(f"{k:20s}: default_modality_mask [C, E, L, S] = {mask}")
print("KAGGLE PREPROCESSING PIPELINE VERIFICATION READY.")
